In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_csv('../datasets/dataset_clean.csv')
print("Setup ready!")

df.head(5)

Matplotlib is building the font cache; this may take a moment.


Setup ready!


,record_id,posisi_pekerjaan,level_senioritas,tahun_pengalaman_kerja,gaji_pokok_bulanan_idr,total_kompensasi_tahunan_idr,skala_perusahaan,tipe_pendanaan,lokasi_kantor,model_kerja,...,panjang_karakter_ulasan,jumlah_kata_ulasan,timestamp_survei,indeks_stabilitas_finansial,premi_keahlian_ai,risiko_attrition_karyawan,rasio_kompensasi_pengalaman,penalaran_karir_cot,posterior_confidence,posisi_pekerjaan_clean
0,TECH_2026_00001,Cybersecurity Specialist,Junior / Entry-Level,2,14000000,182000000,Startup Growth Series B/C,PMA Modal Ventura Asing,Jakarta Selatan / SCBD,100% WFH Remote,...,300,40,2024-01-25 10:17:28,0.73,0.81,Sedang,91000000,Profesional 'Cybersecurity Specialist' pada le...,0.938,Cybersecurity Specialist
1,TECH_2026_00002,Fullstack Engineer,Senior,7,31500000,409500000,Startup Growth Series B/C,PMA Modal Ventura Asing,Bali,100% WFH Remote,...,274,36,2026-09-17 21:23:25,0.56,0.43,Kritis,58500000,Profesional 'Fullstack Engineer' pada level 'S...,0.985,Fullstack Engineer
2,TECH_2026_00003,QA Automation Engineer,Lead / Principal,10,68500000,976204846,Decacorn / Unicorn Lokal,PMA Modal Ventura Asing,Jakarta Selatan / SCBD,100% WFH Remote,...,212,28,2024-10-07 04:30:40,0.52,0.73,Kritis,97620485,Profesional 'QA Automation Engineer' pada leve...,0.990,QA / Software Tester
3,TECH_2026_00004,DevOps / SRE,Senior,6,54000000,756000000,Decacorn / Unicorn Lokal,Korporasi Terbuka Tbk,Luar Negeri / Remote Global,Hybrid 2-3 Hari Kantor,...,214,24,2026-06-14 10:00:17,0.73,0.97,Rendah,126000000,Profesional 'DevOps / SRE' pada level 'Senior'...,0.986,DevOps / SRE
4,TECH_2026_00005,Technical Writer,Mid-Level,5,31000000,403000000,Startup Growth Series B/C,PMDN Modal Ventura Lokal,Jakarta Selatan / SCBD,100% WFH Remote,...,191,25,2024-07-03 16:00:57,0.66,0.58,Kritis,80600000,Profesional 'Technical Writer' pada level 'Mid...,0.990,Technical Writer


In [3]:
selected_features = [
    'posisi_pekerjaan_clean',
    'lokasi_kantor',
    'skala_perusahaan',
    'tahun_pengalaman_kerja'
]

X = df[selected_features].copy()
y = df['gaji_pokok_bulanan_idr']

categorical_features = ['posisi_pekerjaan_clean', 'lokasi_kantor', 'skala_perusahaan']
for col in categorical_features:
    X[col] = X[col].astype(str)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"R-squared (Akurasi Model): {r2_score(y_test, y_pred):.2f}")
print(f"Rata-rata Selisih (MAE): Rp {mean_absolute_error(y_test, y_pred):,.0f}")

R-squared (Akurasi Model): 0.84
Rata-rata Selisih (MAE): Rp 7,018,610


In [4]:
posisi_list = sorted(df['posisi_pekerjaan_clean'].unique().tolist())
lokasi_list = sorted(df['lokasi_kantor'].unique().tolist())
skala_list = sorted(df['skala_perusahaan'].unique().tolist())

posisi_dropdown = widgets.Dropdown(options=posisi_list, description='Posisi:')
lokasi_dropdown = widgets.Dropdown(options=lokasi_list, description='Lokasi:')
skala_dropdown = widgets.Dropdown(options=skala_list, description='Skala Perw:')
pengalaman_slider = widgets.IntSlider(value=3, min=0, max=15, step=1, description='Pengalaman:')

submit_btn = widgets.Button(description='Hitung & Lihat Grafik', button_style='success')
output_area = widgets.Output()

def on_submit_clicked(b):
    with output_area:
        clear_output(wait=True)
        
        import matplotlib.ticker as ticker

        user_posisi = posisi_dropdown.value
        user_lokasi = lokasi_dropdown.value
        user_skala = skala_dropdown.value
        user_exp = pengalaman_slider.value
        
        # Buat sampel data dengan tipe string yang konsisten
        sample_data = pd.DataFrame({
            'posisi_pekerjaan_clean': [str(user_posisi)],
            'tahun_pengalaman_kerja': [int(user_exp)],
            'lokasi_kantor': [str(user_lokasi)],
            'skala_perusahaan': [str(user_skala)]
        })
        
        # Prediksi aman via Pipeline
        pred_salary = model.predict(sample_data)[0]
        
        print("--------------------------------------------------")
        print(f"HASIL ESTIMASI GAJI POKOK BULANAN:")
        print(f"-> Rp {pred_salary:,.0f}")
        print("--------------------------------------------------\n")
        
        # Visualisasi Grafik
        plt.figure(figsize=(10, 5))
        
        ax = plt.gca()
        ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f'{x*1e-6:.0f} Jt'))

        subset_df = df[(df['posisi_pekerjaan_clean'] == user_posisi) & (df['lokasi_kantor'] == user_lokasi)]
        if len(subset_df) < 5: 
            subset_df = df[df['posisi_pekerjaan_clean'] == user_posisi]
            
        sns.scatterplot(
            data=subset_df, 
            x='tahun_pengalaman_kerja', 
            y='gaji_pokok_bulanan_idr', 
            hue='skala_perusahaan', 
            palette='Set2', 
            s=80, 
            alpha=0.8
        )
        
        sns.regplot(
            data=subset_df, 
            x='tahun_pengalaman_kerja', 
            y='gaji_pokok_bulanan_idr', 
            scatter=False, 
            color='black', 
            line_kws={'linestyle':'--', 'linewidth':2, 'label':'Tren Linear (Regresi)'}
        )
        
        plt.scatter([user_exp], [pred_salary], color='red', s=150, zorder=5, marker='*', label='Estimasi Pilihan Anda')

        import matplotlib.ticker as ticker
        ax = plt.gca()

        ax.yaxis.set_major_locator(ticker.MultipleLocator(10000000))
        
        plt.title(f"Grafik Estimasi Gaji: {user_posisi} di {user_lokasi}", fontsize=12, fontweight='bold')
        plt.xlabel("Tahun Pengalaman Kerja", fontsize=10)
        plt.ylabel("Gaji Pokok Bulanan (IDR)", fontsize=10)
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(True, linestyle=':', alpha=0.6)
        plt.tight_layout()
        plt.show()

submit_btn.on_click(on_submit_clicked)
display(widgets.VBox([posisi_dropdown, lokasi_dropdown, skala_dropdown, pengalaman_slider, submit_btn, output_area]))